# 04 — Machine Learning: Fare Prediction

**Tickets:** ML-01, ML-02, ML-03, ML-04, ML-05, ML-06, ML-07  
**Business Question (BQ-3):** Can we predict the total fare of a trip before it starts?  
**Purpose:** Build a feature table, train baseline and improved models, log everything to MLflow, register the best model.

---

## Setup

## ML-01 — Algorithm Research & Selection

### Problem Framing

**Task type:** Supervised regression — predict `total_amount` (continuous target) from features known at trip start.

**Features available before trip starts (from Silver table):**
| Feature | Type | Notes |
|---------|------|-------|
| `PULocationID` | Categorical (265 zones) | Pickup taxi zone — encode as one-hot or target-encode |
| `DOLocationID` | Categorical (265 zones) | Dropoff taxi zone (if known at dispatch) |
| `hour_of_day` | Ordinal (0-23) | Extracted from `tpep_pickup_datetime` |
| `day_of_week` | Ordinal (0-6) | Mon=0, Sun=6 |
| `is_weekend` | Binary | Saturday/Sunday flag |
| `trip_distance` | Continuous | Miles — strong predictor, but verify no data leakage |
| `passenger_count` | Discrete (1-9) | Weak predictor, but include for completeness |
| `RatecodeID` | Categorical (1-6) | Standard, JFK, Newark, etc. — very informative |

**Target:** `total_amount` (includes fare, surcharges, tips, tolls)

> **Leakage warning:** `fare_amount`, `tip_amount`, `tolls_amount`, `mta_tax`, `improvement_surcharge` are all components of `total_amount` — they must NOT be used as features.

---

### Candidate Algorithms

| # | Algorithm | Library | Pros | Cons | Expected role |
|---|-----------|---------|------|------|---------------|
| 1 | **Linear Regression** | `sklearn.linear_model.LinearRegression` | Fast to train, fully interpretable, good baseline to measure uplift against | Assumes linear feature-target relationship; struggles with interactions and non-linearity | **Baseline (ML-03)** |
| 2 | **Gradient Boosted Trees (GBT)** | `sklearn.ensemble.HistGradientBoostingRegressor` | Handles non-linearity & interactions natively; usually top performer on tabular data; feature importance built in | Slower to train; more hyperparams to tune; risk of overfitting with small data | **Improved model (ML-04)** |
| 3 | **Random Forest** | `sklearn.ensemble.RandomForestRegressor` | Robust out-of-box; less prone to overfitting than single-tree GBT; parallelisable | Typically slightly worse than tuned GBT; larger model size | **Alternative improved model or stretch (S-06)** |

#### Why these three?

- The project plan explicitly names LinearRegression, GBT, and RF as candidates.
- They cover a clear **baseline  to  improved** progression: linear  to  ensemble.
- All available via `scikit-learn` (already in `requirements.txt`), no extra dependencies needed.
- All produce feature importances (coefficients / impurity-based), useful for ML-07 interpretation.
- `HistGradientBoostingRegressor` is preferred over `GradientBoostingRegressor` for larger datasets (>10k rows) — it uses histogram-based binning for dramatically faster training with near-identical accuracy.

#### Stretch candidates (S-06)

| Algorithm | When to consider |
|-----------|-----------------|
| **Ridge / Lasso** | If Linear Regression overfits or we want regularisation |
| **XGBoost** | If `HistGradientBoosting` is not enough and we want more tuning control (requires extra install) |
| **LightGBM** | Databricks-native; fastest GBT variant; ideal if dataset is very large |

---

### Evaluation Metrics

| Metric | Why |
|--------|-----|
| **RMSE** | Primary metric — penalises large errors (important for fare estimates) |
| **MAE** | Interpretable "average dollar error" |
| **R-squared** | How much variance the model explains vs mean baseline |

All three are logged to MLflow per the project plan (ML-05).

---

### Recommended Approach

1. **ML-02:** Build feature table from Silver — select the columns above, encode categoricals, train/test split (80/20, random or stratified by `hour_of_day`).
2. **ML-03:** Train `LinearRegression` as baseline. Log to MLflow.
3. **ML-04:** Train `HistGradientBoostingRegressor` with light hyperparameter search (`max_iter`, `max_depth`, `learning_rate`). Log to MLflow.
4. **ML-05:** Compare both runs in MLflow — pick the lower-RMSE model.
5. **ML-06:** Register winner in MLflow Model Registry.
6. **ML-07:** Extract feature importances; write interpretation narrative.

If time allows, add Random Forest and/or XGBoost as stretch comparisons (S-06).

In [ ]:
# TODO: initialise SparkSession / Databricks context
# import mlflow
# import mlflow.sklearn

## ML-02 — Build feature table

In [ ]:
# TODO: read Silver table
# TODO: select features: pickup_location, hour_of_day, day_of_week, trip_distance, passenger_count
# TODO: target: total_amount
# TODO: train/test split

## ML-03 — Baseline model (Linear Regression)

In [ ]:
# TODO: train LinearRegression baseline
# TODO: log params & metrics (RMSE, MAE, R²) to MLflow

## ML-04 — Improved model (Gradient Boosted Trees)

In [ ]:
# TODO: train GBT / Random Forest
# TODO: log params & metrics to MLflow

## ML-05 — Compare models in MLflow

In [ ]:
# TODO: print comparison table of RMSE / MAE / R² for both models

## ML-06 — Register best model

In [ ]:
# TODO: register best model to MLflow Model Registry

## ML-07 — Model interpretation

<!-- Summarise what the model learned: top features, directional relationships, limitations -->